# 🚀 从零搭建个人博客 — 全栈项目教程

> 面向编程初学者，手把手讲解一个完整 Serverless 博客项目的所有知识点。

## 学完这节课，你将掌握：

| 层次 | 技术 | 在本项目中的角色 |
|:---|:---|:---|
| 🎨 前端 | HTML + CSS + JavaScript | 页面结构、样式、交互逻辑 |
| ✍️ 编辑器 | Quill.js（富文本编辑器） | 让管理员像 Word 一样编辑文章 |
| 🗄️ 后端 | Supabase（PostgreSQL 数据库） | 存储和读取文章数据 |
| 🔐 认证 | 前端管理密码 + localStorage | 区分访客和管理员 |
| 🌐 部署 | GitHub Pages | 免费托管静态网站 |

> 📁 项目代码位于 `20260712/blog-demo/index.html`，配套指南位于 `SUPABASE-SETUP-GUIDE.md`。

---

## 1. 项目整体架构

### 1.1 什么是 Serverless？

**Serverless**（无服务器）不是说"没有服务器"，而是说**你不需要自己管理服务器**。

传统方式 vs Serverless：

```
┌─ 传统方式 ──────────────────────────────────┐
│  你需要：                                     │
│  1. 租一台云服务器（如阿里云、AWS）             │
│  2. 安装操作系统、配置网络                      │
│  3. 安装数据库（MySQL / PostgreSQL）           │
│  4. 写后端代码（Python Flask / Node.js）       │
│  5. 配置 Nginx 反向代理                       │
│  6. 维护、更新、打补丁…                        │
│  💰 每月几十到几百元                           │
└──────────────────────────────────────────────┘

┌─ Serverless 方式（本项目）───────────────────┐
│  你只需要：                                    │
│  1. 一个 HTML 文件（前端）                     │
│  2. 一个 Supabase 账号（后端数据库）            │
│  3. 把它们连起来                               │
│  💰 免费                                       │
└──────────────────────────────────────────────┘
```

### 1.2 本项目的架构图

```
┌──────────────┐     Supabase JS SDK     ┌─────────────────┐
│  浏览器       │ ◄──────────────────────► │  Supabase 云端   │
│              │    (HTTPS + anon key)    │                 │
│  index.html  │                          │  PostgreSQL     │
│  ├─ HTML 结构 │                          │  └─ posts 表    │
│  ├─ CSS 样式  │                          │                 │
│  ├─ JS 逻辑   │                          │  Quill.js CDN   │
│  └─ Quill 编辑器│                         │  Google Fonts   │
│              │                          │                 │
│  托管于：      │                          │                 │
│  GitHub Pages │                          │                 │
└──────────────┘                          └─────────────────┘
```

**关键点**：
- 前端是**一个文件**（`index.html`），包含 HTML + CSS + JS
- 数据库在 Supabase 云端，通过 JS SDK 直接连接
- 没有中间服务器 — 浏览器直接和数据库对话
- 部署在 GitHub Pages，完全免费

---

## 2. HTML 基础 — 页面的骨架

### 2.1 HTML 是什么？

HTML（HyperText Markup Language，超文本标记语言）是网页的**骨架**。它用标签（tag）来描述页面的结构。

```html
<标签名 属性="值">内容</标签名>
```

比如：`<h1>我的博客</h1>` 表示一个一级标题。

### 2.2 项目中用到的 HTML 标签

让我们看看 `index.html` 中的核心结构：

In [ ]:
# 这不是可运行的代码，而是展示 HTML 结构的示意图
# 实际代码在 index.html 中

html_structure = """
<!DOCTYPE html>                    ← 声明这是 HTML5 文档
<html lang="zh-CN">               ← 根元素，lang 表示页面语言
<head>                             ← 头部：元信息、样式、脚本
  <meta charset="UTF-8">          ← 字符编码（支持中文）
  <meta name="viewport" ...>      ← 移动端适配
  <title>我的博客</title>           ← 浏览器标签页标题
  <link href="...css">            ← 引入外部 CSS
</head>
<body>                             ← 身体：可见内容
  <header class="masthead">       ← 顶部导航栏
    <a class="site-title">...</a>  ← 博客标题（可点击）
    <button>⚙</button>            ← 管理按钮
  </header>
  <div class="page-wrap">         ← 页面主体容器
    <aside class="sidebar">       ← 侧边栏
      <div class="bio-block">     ← 个人信息卡片
        <div class="bio-avatar">  ← 头像
        <div class="bio-name">    ← 名字
        ...
      </div>
      <div class="admin-panel">   ← 管理面板（默认隐藏）
        <input id="postTitle">    ← 文章标题输入框
        <div id="editor-container">← Quill 富文本编辑器
        <button>发布</button>      ← 提交按钮
      </div>
    </aside>
    <main>                         ← 主内容区
      <ul class="feed-list">      ← 文章列表容器
        <!-- 文章由 JS 动态生成 -->
      </ul>
    </main>
  </div>
</body>
</html>
"""

print("📐 HTML 结构示意：")
print(html_structure)

### 2.3 关键标签详解

| 标签 | 作用 | 在项目中的位置 |
|:---|:---|:---|
| `<header>` | 页面头部（导航栏） | 顶部粘性导航 `class="masthead"` |
| `<aside>` | 侧边栏（辅助内容） | 个人信息 + 管理面板 |
| `<main>` | 主内容区 | 文章列表 |
| `<div>` | 通用容器（最常用） | 几乎所有布局块 |
| `<input>` | 输入框 | 文章标题输入 `id="postTitle"` |
| `<button>` | 按钮 | 发布 / 取消 / 管理入口 |
| `<ul>` + `<li>` | 无序列表 + 列表项 | 文章列表 `class="feed-list"` |

### 2.4 `id` vs `class` — 什么时候用哪个？

- **`id`**：页面中**唯一**的标识符，JS 用 `document.getElementById('xxx')` 获取
- **`class`**：可以**多个元素共享**的样式名，CSS 用 `.class名` 选择

```html
<!-- id → JS 用来找到这个元素 -->
<input id="postTitle" placeholder="标题…">

<!-- class → CSS 用来设置样式 -->
<div class="post-item">...</div>
<div class="post-item">...</div>  <!-- 可以重复 -->
```

---

## 3. CSS 基础 — 页面的皮肤

### 3.1 CSS 是什么？

CSS（Cascading Style Sheets，层叠样式表）是网页的**外观**。它控制颜色、字体、间距、布局。

```css
选择器 {
  属性: 值;
}
```

比如：`h1 { color: red; }` 让所有一级标题变红。

### 3.2 本项目的 CSS 设计系统

项目使用 **CSS 变量（Custom Properties）** 来统一管理颜色和字体：

In [ ]:
# CSS 变量（CSS Custom Properties）示意
print("🎨 项目的设计系统（Design Tokens）：")
print()

tokens = {
    "--ink":   "#1a1a18",   # 墨色 — 主文字颜色
    "--paper": "#f7f4ee",   # 纸色 — 页面背景
    "--mist":  "#ede9e0",   # 雾色 — 边框、分隔线
    "--stone": "#8a8478",   # 石色 — 次要文字
    "--rust":  "#b85c38",   # 锈色 — 强调色（链接、hover）
    "--gold":  "#c9a84c",   # 金色 — 备用强调色
    "--serif": "'Playfair Display', 'Noto Serif SC', serif",  # 衬线字体
    "--mono":  "'JetBrains Mono', monospace",                  # 等宽字体
    "--body":  "'Noto Serif SC', Georgia, serif",              # 正文字体
}

for k, v in tokens.items():
    print(f"  {k:12s} → {v}")

print()
print("💡 为什么用 CSS 变量？")
print("  改一个 --rust 颜色，所有用到它的地方自动更新，不用全局搜索替换。")

### 3.3 布局：CSS Grid（网格布局）

本项目的页面使用 **CSS Grid** 实现左右两栏布局：

```css
.page-wrap {
  display: grid;
  grid-template-columns: 260px 1fr;  /* 左侧 260px，右侧占满剩余空间 */
  gap: 60px;                          /* 两栏间距 60px */
}
```

```
┌──────────────┬──────────────────────────────────┐
│   Sidebar    │          Feed（文章列表）          │
│   260px      │          1fr（剩余空间）           │
│              │                                  │
│  [头像]       │  ┌─ 文章卡片 ──────────────────┐ │
│  名字         │  │ 日期 │ 标题                  │ │
│  学校         │  │      │ 内容…                │ │
│  标签         │  └────────────────────────────┘ │
│              │  ┌─ 文章卡片 ──────────────────┐ │
│  文章数       │  │ 日期 │ 标题                  │ │
│              │  │      │ 内容…                │ │
│  (管理面板)   │  └────────────────────────────┘ │
└──────────────┴──────────────────────────────────┘
```

同样地，每篇文章卡片也用 Grid 排版（日期在左，内容在右）：

```css
.post-item {
  display: grid;
  grid-template-columns: 80px 1fr;  /* 日期 80px，内容占满剩余 */
  gap: 0 24px;                       /* 行间距 0，列间距 24px */
}
```

### 3.4 响应式设计 — 手机也能看

**响应式设计（Responsive Design）** 让网站在手机和电脑上都能正常显示。核心工具是 **media query**：

```css
@media (max-width: 720px) {
  /* 当屏幕宽度 ≤ 720px（手机）时，应用这些样式 */
  .page-wrap {
    grid-template-columns: 1fr;  /* 从两栏变成单栏 */
  }
  .sidebar {
    position: relative;  /* 取消 sticky 定位 */
  }
}
```

| 设备 | 屏幕宽度 | 布局 |
|:---|:---|:---|
| 💻 电脑 | > 720px | 左右两栏（Grid 260px + 1fr）|
| 📱 手机 | ≤ 720px | 上下单栏（Grid 1fr）|

### 3.5 其他关键 CSS 技术

| 技术 | 代码 | 作用 |
|:---|:---|:---|
| **粘性定位** | `position: sticky; top: 0;` | 顶部导航栏滚动时固定不动 |
| **CSS 过渡** | `transition: all 0.2s ease;` | 按钮 hover 时平滑变色 |
| **CSS 动画** | `@keyframes fadeUp { ... }` | 文章卡片淡入上浮 |
| **SVG 纹理** | `background-image: url("data:image/svg...")` | 纸张纹理效果 |

---

## 4. JavaScript 基础 — 页面的"大脑"

### 4.1 JavaScript 在项目中的角色

HTML = 骨架，CSS = 皮肤，**JavaScript = 大脑和肌肉**。它负责：

| 功能 | 对应代码 |
|:---|:---|
| 从数据库加载文章 | `fetchPosts()` |
| 渲染文章列表 | `renderFeed(posts)` |
| 管理登录/退出 | `toggleAdmin()` |
| 新建/编辑/删除文章 | `handleSubmit()` / `editPost()` / `deletePost()` |
| 恢复默认数据 | `restoreDefaultData()` |

### 4.2 DOM 操作 — JS 如何操控 HTML

**DOM（Document Object Model）** 是浏览器把 HTML 解析后的"对象树"。JS 通过 DOM API 来读写页面内容。

```javascript
// 获取元素（通过 id）
const titleInput = document.getElementById('postTitle');

// 读取输入框的值
const title = titleInput.value;

// 修改元素的文字
document.getElementById('postCountNum').textContent = '5';

// 修改元素的 HTML
document.getElementById('feedList').innerHTML = '<div>新内容</div>';

// 显示/隐藏元素
document.getElementById('adminPanel').style.display = 'block';  // 显示
document.getElementById('adminPanel').style.display = 'none';   // 隐藏

// 添加/移除 CSS 类
element.classList.add('visible');    // 添加类
element.classList.remove('visible'); // 移除类
element.classList.toggle('visible', true);  // 条件切换
```

In [ ]:
# JavaScript DOM 操作概念演示（用 Python 模拟）

print("🖥️ JS 的 DOM 操作是浏览器专属能力，")
print("  这里用 Python 字典模拟一个简单的页面状态：")
print()

# 模拟页面状态
page = {
    "postTitle": "",           # 标题输入框的值
    "postCountNum": "—",       # 文章数量显示
    "adminPanel_visible": False, # 管理面板是否可见
    "feedList": []              # 文章列表
}

print("初始状态:", page)

# 模拟用户输入标题
page["postTitle"] = "我的第一篇文章"
print("\n输入标题后:", page["postTitle"])

# 模拟加载文章
page["feedList"] = [{"title": "你好世界", "content": "<p>欢迎</p>"}]
page["postCountNum"] = str(len(page["feedList"]))
print("\n加载文章后: 共", page["postCountNum"], "篇")

# 模拟切换管理模式
page["adminPanel_visible"] = not page["adminPanel_visible"]
print("\n切换管理: adminPanel =", "显示" if page["adminPanel_visible"] else "隐藏")

### 4.3 异步编程 — `async` / `await`

当你向 Supabase 请求数据时，需要**等待**服务器回复。JS 用 `async/await` 来处理这种"等一会儿"的操作：

```javascript
// async 表示"这个函数里面有等待操作"
async function fetchPosts() {
  // await 表示"在这里等一下，拿到了结果再继续"
  const { data, error } = await sb
    .from('posts')       // 选择 posts 表
    .select('*')         // 查询所有列
    .order('id', { ascending: false });  // 按 id 降序

  if (error) {
    console.error('出错了:', error.message);
  } else {
    console.log('拿到了', data.length, '篇文章');
  }
}
```

**为什么要 async/await？**

网络请求需要时间（几百毫秒）。如果不用 await，代码会在数据还没回来时就继续往下跑，导致 `data` 为 `undefined`。

### 4.4 动态渲染 — 从数据到 HTML

项目不预先写好每篇文章的 HTML，而是从数据库拿到数据后**用 JS 动态生成** HTML。这叫做 **客户端渲染（CSR）**。

```javascript
function renderFeed(posts) {
  const list = document.getElementById('feedList');
  list.innerHTML = '';  // 先清空

  posts.forEach((post, i) => {
    // 为每篇文章创建一个 <li> 元素
    const li = document.createElement('li');
    li.className = 'post-item';
    li.style.animationDelay = `${i * 60}ms`;  // 每个延迟不同，产生依次出现的效果

    // 用模板字符串填充内容
    li.innerHTML = `
      <div class="post-date-col">
        <span class="post-month">JAN</span>
        <span class="post-day">01</span>
      </div>
      <div class="post-body">
        <h2 class="post-title">${post.title}</h2>
        <div class="post-excerpt">${post.content}</div>
      </div>`;

    list.appendChild(li);  // 添加到页面上
  });
}
```

> `${变量}` 是 **模板字符串（Template Literal）**，用反引号 `` ` `` 包裹，`${}` 里面可以放任何 JS 表达式。

In [ ]:
# Python 模拟：模板字符串 + 动态渲染

posts = [
    {"title": "你好，世界！", "date": "2026-01-01", "content": "<p>欢迎来到我的博客！</p>"},
    {"title": "关于本站", "date": "2026-01-02", "content": "<p>技术栈介绍…</p>"},
    {"title": "我的项目", "date": "2026-01-03", "content": "<p>项目链接…</p>"},
]

print("📄 动态渲染的文章列表（模拟 JS 的 innerHTML 注入）：")
print("=" * 50)

for i, post in enumerate(posts, 1):
    # 模拟 JS 中的模板字符串 `${post.title}` 等
    card = f"""
┌─ 文章 {i} ─────────────────────────────┐
│ 📅 {post['date']}
│ 📝 标题：{post['title']}
│ 📄 内容：{post['content']}
└────────────────────────────────────────┘"""
    print(card)

print(f"\n共 {len(posts)} 篇文章（由 JS 动态生成，不是写死在 HTML 里的）")

### 4.5 集中配置 — CONFIG 对象

项目把所有可配置内容集中在一个 `CONFIG` 对象中：

```javascript
const CONFIG = {
  supabaseUrl:  'https://YOUR-PROJECT-ID.supabase.co',
  supabaseKey:  'YOUR-ANON-KEY',
  adminPass:    'admin',
  siteTitle:    '我的博客',
  subtitle:     '— 个人空间',
  avatarSeed:   'Blog',
  bioName:      '你的名字',
  bioSchool:    '你的学校 / 组织',
  bioText:      '一句话介绍自己<br>兴趣爱好、专业领域',
  bioTags:      ['标签1', '标签2', '标签3'],
  defaultPosts: [ /* 3 篇文章模板 */ ],
  postTag:      '# echo'
};
```

**为什么这样做？** 改一处，全局生效。标题改了，浏览器标签、页面标题、所有引用处自动更新。这种模式叫 **DRY（Don't Repeat Yourself，不重复自己）**。

---

## 5. Supabase — 云端数据库

### 5.1 什么是 Supabase？

**Supabase** 是一个 **Backend-as-a-Service（后端即服务）** 平台。它提供：

| 服务 | 说明 |
|:---|:---|
| 🗄️ PostgreSQL 数据库 | 完整的关系型数据库，支持 SQL |
| 🔐 认证服务 | 邮箱/手机/第三方登录（本项目未使用） |
| 📡 自动 REST API | 建表后自动生成增删改查 API |
| 🔒 行级安全（RLS） | 控制谁能读写哪些数据 |

> 💡 它是开源的，可以自托管，也可以直接用官方云服务（免费额度足够个人项目）。

### 5.2 数据库表设计

本项目的核心是一张 `posts` 表：

In [ ]:
# SQL 建表语句（在 Supabase SQL Editor 中执行）

create_table_sql = """
-- ========== 创建 posts 表 ==========
CREATE TABLE IF NOT EXISTS posts (
  id         BIGINT       PRIMARY KEY,        -- 文章 ID（用 Date.now() 生成）
  title      TEXT         NOT NULL,            -- 文章标题
  content    TEXT         NOT NULL DEFAULT '', -- 文章正文（Quill 输出的 HTML）
  date       TEXT         DEFAULT '',          -- 显示日期，如 "2026年7月11日"
  raw_date   TEXT         DEFAULT '',          -- ISO 日期，如 "2026-07-11"
  created_at TIMESTAMPTZ  DEFAULT NOW()        -- 数据库自动时间戳
);

-- ========== 索引（加速查询） ==========
CREATE INDEX IF NOT EXISTS idx_posts_id ON posts(id DESC);
"""

print("📋 posts 表结构：")
print(create_table_sql)

# 表结构说明
import pandas as pd
schema = pd.DataFrame([
    ["id", "BIGINT", "主键，用 Date.now() 生成毫秒时间戳（如 1000000000001）"],
    ["title", "TEXT", "文章标题，如 '你好，世界！'"],
    ["content", "TEXT", "文章正文，存 Quill 编辑器输出的 HTML"],
    ["date", "TEXT", "中文格式日期，如 '2026年7月11日'"],
    ["raw_date", "TEXT", "ISO 格式日期，如 '2026-07-11'，用于排序"],
    ["created_at", "TIMESTAMPTZ", "数据库自动填充的创建时间戳"],
], columns=["字段", "类型", "说明"])

print("\n字段说明：")
print(schema.to_string(index=False))

### 5.3 前端如何操作数据库？

Supabase 提供 JS SDK，让浏览器直接操作数据库，无需后端服务器。

#### 读取（SELECT）

```javascript
// 读取所有文章，按 id 降序（最新在前）
const { data, error } = await sb
  .from('posts')
  .select('*')
  .order('id', { ascending: false });
```

#### 创建（INSERT）

```javascript
const { error } = await sb.from('posts').insert([{
  id: Date.now(),
  title: '新文章',
  content: '<p>正文 HTML</p>',
  date: '2026年7月11日',
  raw_date: '2026-07-11'
}]);
```

#### 更新（UPDATE）

```javascript
const { error } = await sb
  .from('posts')
  .update({ title: '修改后的标题', content: '<p>新内容</p>' })
  .eq('id', 1000000000001);  // WHERE id = 1000000000001
```

#### 删除（DELETE）

```javascript
const { error } = await sb
  .from('posts')
  .delete()
  .eq('id', 1000000000001);
```

> 这些操作对应 SQL 中的 **CRUD**：Create（创建）、Read（读取）、Update（更新）、Delete（删除）。

In [ ]:
# Python 模拟：CRUD 操作（对应 Supabase JS SDK 的链式调用）

print("🗄️ Supabase JS SDK 的 CRUD 操作（概念演示）")
print("=" * 50)
print()

# 模拟数据库
db_posts = []

# CREATE — 创建文章
new_post = {"id": 1000000000001, "title": "你好世界", "content": "<p>欢迎</p>", "date": "2026-07-11"}
db_posts.append(new_post)
print("✅ CREATE: 新增文章 →", new_post["title"])

# READ — 读取所有文章
print(f"✅ READ:   当前共 {len(db_posts)} 篇文章")
for p in db_posts:
    print(f"         - [{p['id']}] {p['title']}")

# UPDATE — 更新文章
for p in db_posts:
    if p["id"] == 1000000000001:
        p["title"] = "你好世界（已修改）"
        break
print(f"✅ UPDATE: 标题改为 → {db_posts[0]['title']}")

# DELETE — 删除文章
db_posts = [p for p in db_posts if p["id"] != 1000000000001]
print(f"✅ DELETE: 删除后剩余 {len(db_posts)} 篇文章")

print()
print("💡 JS SDK 的链式调用风格（如 .from().select().order()）")
print("   叫做 Fluent Interface（流式接口），读起来像英语句子。")

### 5.4 认证令牌（Auth Token）

本项目的安全模型**不是** Supabase Auth（邮箱密码登录），而是**前端管理密码**：

```
┌─────────────┐                    ┌─────────────┐
│   访客       │                    │   管理员     │
│             │                    │             │
│ 看到文章列表  │  点击 ⚙ 输入密码    │ 看到编辑面板  │
│ 无编辑权限   │ ──────────────────► │ 可增删改文章  │
│             │   localStorage      │             │
│             │   is_admin_mode='1'  │             │
└─────────────┘                    └─────────────┘
```

**Token 的概念**（即使本项目不直接使用 JWT）：

- **Anon Key**：公开的"匿名令牌"，前端用它向 Supabase 证明"我来自你的项目"
- **Service Role Key**：私密的"管理员令牌"，绕过所有权限，**绝不放在前端**

```javascript
// 初始化 Supabase 客户端，anon key 作为"身份标识"
const sb = window.supabase.createClient(
  CONFIG.supabaseUrl,   // 你的 Supabase 项目地址
  CONFIG.supabaseKey    // 你的 anon key（公开的令牌）
);

// 之后所有 .from('posts').select() 等调用
// 都会自动在 HTTP 请求头中带上这个 key
```

> 在真正的 JWT 认证中，用户登录后会获得一个 token，前端每次发请求时在 `Authorization: Bearer <token>` 头中携带。本项目简化了这一流程，用管理密码 + localStorage 代替。

---

## 6. REST API 概念

### 6.1 什么是 API？

**API（Application Programming Interface，应用程序接口）** 是两个程序之间"对话"的约定。

就像餐厅里：你是顾客（前端），厨房是后端，**服务员（API）** 是你和厨房之间的桥梁：
- 你告诉服务员"我要一份炒饭"（请求）
- 服务员去厨房下单
- 服务员把炒饭端给你（响应）

### 6.2 什么是 REST？

**REST（Representational State Transfer）** 是一种 API 设计风格，用 HTTP 方法表示操作意图：

| HTTP 方法 | 操作 | Supabase 对应代码 |
|:---|:---|:---|
| `GET` | 读取 | `.select('*')` |
| `POST` | 创建 | `.insert([...])` |
| `PATCH` | 更新 | `.update({...})` |
| `DELETE` | 删除 | `.delete()` |

### 6.3 本项目中的 API 端点

Supabase **自动为每张表生成 REST API**。当你执行 `.from('posts').select('*')` 时，底层实际发送的 HTTP 请求是：

```
GET https://YOUR-PROJECT-ID.supabase.co/rest/v1/posts?select=*
Headers:
  apikey: YOUR-ANON-KEY
  Authorization: Bearer YOUR-ANON-KEY
```

你不需要手动写这些 HTTP 请求 — Supabase JS SDK 帮你封装好了。但理解底层原理很重要！

In [ ]:
# Python 模拟 REST API 调用（实际项目用 Supabase JS SDK，这里展示概念）

print("🌐 REST API 请求/响应 模拟")
print("=" * 50)
print()

# 模拟：前端通过 Supabase JS SDK 请求数据时，底层发生了什么

print("📤 请求（Request）：")
print("  方法:   GET")
print("  地址:   https://xxx.supabase.co/rest/v1/posts?select=*")
print("  请求头: apikey: eyJhbGciOi...（你的 anon key）")
print("         Authorization: Bearer eyJhbGciOi...")
print()

print("📥 响应（Response）：")
print("  状态码: 200 OK")
print("  响应体:")

import json
response_body = [
    {"id": 1000000000001, "title": "你好，世界！", "date": "2026-01-01", "raw_date": "2026-01-01"},
    {"id": 1000000000002, "title": "关于本站", "date": "2026-01-01", "raw_date": "2026-01-01"},
]
print(json.dumps(response_body, indent=2, ensure_ascii=False))

print()
print("💡 这些 HTTP 细节由 Supabase JS SDK 自动处理，")
print("   你只需写 .from('posts').select('*') 即可。")

---

## 7. Quill.js — 富文本编辑器

### 7.1 为什么不用普通的 `<textarea>`？

普通的 `<textarea>` 只能输入**纯文本**（没有加粗、斜体、列表等格式）。

**Quill.js** 是一个开源富文本编辑器，提供类似 Word 的编辑体验，输出的是 **HTML**。

### 7.2 如何集成到项目中？

只需要两步：

**第一步：引入 Quill 的 CSS 和 JS（通过 CDN）**

```html
<!-- 在 <head> 中引入 CSS -->
<link href="https://cdnjs.cloudflare.com/ajax/libs/quill/1.3.7/quill.snow.min.css" rel="stylesheet">

<!-- 在页面加载时引入 JS（带 fallback） -->
<script src="https://cdnjs.cloudflare.com/ajax/libs/quill/1.3.7/quill.min.js"></script>
```

**第二步：初始化编辑器**

```javascript
quill = new Quill('#editor-container', {
  theme: 'snow',
  placeholder: '写点什么…',
  modules: {
    toolbar: [
      ['bold', 'italic', 'underline'],  // 加粗、斜体、下划线
      ['blockquote', 'code-block'],      // 引用块、代码块
      [{ 'list': 'bullet' }],            // 无序列表
      ['link', 'image']                  // 链接、图片
    ]
  }
});

// 获取编辑器的 HTML 内容
const htmlContent = quill.root.innerHTML;

// 设置编辑器的内容
quill.root.innerHTML = '<p>已有内容</p>';
```

### 7.3 Quill 在项目中的工作流

```
┌─────────────────┐     ┌──────────┐     ┌──────────┐
│ 管理员打字        │ ──► │ Quill    │ ──► │ Supabase │
│ （加粗、列表等）  │     │ 输出 HTML │     │ posts 表 │
└─────────────────┘     └──────────┘     └──────────┘
                                              │
┌─────────────────┐     ┌──────────┐           │
│ 访客看到格式化文本 │ ◄── │ 浏览器   │ ◄─────────┘
│ （渲染 HTML）     │     │ innerHTML │
└─────────────────┘     └──────────┘
```

---

## 8. localStorage — 浏览器端的"小笔记"

### 8.1 localStorage 是什么？

**localStorage** 是浏览器提供的一个**键值对存储**，数据存在用户的电脑上，即使关闭浏览器也不会丢失。

```javascript
// 存数据
localStorage.setItem('is_admin_mode', '1');

// 取数据
const mode = localStorage.getItem('is_admin_mode');  // → '1'

// 删数据
localStorage.removeItem('is_admin_mode');
```

### 8.2 在本项目中的作用

项目用 localStorage 来**记住管理员登录状态**：

```javascript
function toggleAdmin() {
  const isAdmin = localStorage.getItem('is_admin_mode') === '1';

  if (isAdmin) {
    // 当前是管理员 → 退出
    localStorage.setItem('is_admin_mode', '0');
  } else {
    // 当前是访客 → 弹出密码框
    const pwd = prompt('管理员密码：');
    if (pwd === CONFIG.adminPass) {
      localStorage.setItem('is_admin_mode', '1');  // 记住登录状态
    }
  }
}
```

**为什么用 localStorage 而不是 cookie？**

| 方式 | 容量 | 过期 | 发送到服务器 |
|:---|:---|:---|:---|
| **localStorage** | ~5MB | 永不过期 | ❌ 不发送 |
| **Cookie** | ~4KB | 可设过期 | ✅ 每次请求都发送 |

对于"记住管理模式"这种纯前端状态，localStorage 是最合适的选择。

In [ ]:
# Python 模拟 localStorage 的工作方式

print("💾 localStorage 模拟（键值对存储）")
print("=" * 40)

# 模拟 localStorage
local_storage = {}

# 管理员登录
password_input = "admin"  # 模拟用户输入
if password_input == "admin":
    local_storage["is_admin_mode"] = "1"
    print("✅ 登录成功！is_admin_mode = 1")
else:
    print("❌ 密码错误")

# 页面刷新后（模拟重新加载）
print(f"\n🔄 页面刷新后读取: is_admin_mode = {local_storage.get('is_admin_mode', '0')}")
print(f"   登录状态 {'保留' if local_storage.get('is_admin_mode') == '1' else '丢失'}！")

# 管理员退出
local_storage["is_admin_mode"] = "0"
print(f"\n🚪 退出后: is_admin_mode = {local_storage['is_admin_mode']}")

---

## 9. GitHub Pages 部署

### 9.1 什么是 GitHub Pages？

**GitHub Pages** 是 GitHub 提供的**免费静态网站托管**服务。只要把一个 HTML 文件推到 GitHub 仓库，它就会自动变成一个可以通过网址访问的网站。

### 9.2 部署流程

```bash
# 1. 初始化 Git 仓库
git init

# 2. 添加文件
git add index.html
git commit -m "初始化博客"

# 3. 关联远程仓库
git remote add origin https://github.com/你的用户名/你的仓库.git

# 4. 推送
git push -u origin main
```

然后在 GitHub 上：
1. 进入仓库 → **Settings** → **Pages**
2. **Source** 选择 `main` 分支，目录选 `/ (root)`
3. 点击 **Save**
4. 等待 1-2 分钟，访问 `https://你的用户名.github.io/仓库名/`

### 9.3 为什么可以免费？

因为我们的博客是**纯静态文件**（一个 HTML 文件），不消耗服务器计算资源。GitHub Pages 只负责"把这个文件发给你"，成本极低。

```
┌──────────┐    请求 index.html    ┌──────────────┐
│  访客浏览器 │ ──────────────────► │ GitHub Pages  │
│          │ ◄────────────────── │ (静态文件托管)  │
│          │    返回 index.html    │              │
└──────────┘                      └──────────────┘
                                         │
                                   index.html 中的 JS
                                   自动从 Supabase
                                   加载文章数据
```

---

## 10. 完整数据流 — 从"发布"到"展示"

让我们跟踪一篇文章的完整生命周期：

```
① 管理员点击 ⚙，输入密码
   └─► localStorage.is_admin_mode = '1'
   └─► 侧边栏从个人信息切换为编辑器

② 管理员在 Quill 编辑器中写作
   └─► 输入标题："今天学了什么"
   └─► 编辑内容：<p>今天学习了 <strong>JavaScript</strong></p>

③ 点击 "发布"
   └─► handleSubmit() 被调用
   └─► quill.root.innerHTML 获取 HTML
   └─► sb.from('posts').insert([{...}]) 写入数据库

④ Supabase PostgreSQL 收到 INSERT 请求
   └─► 数据写入 posts 表
   └─► 返回成功响应

⑤ 前端收到成功响应
   └─► fetchPosts() 重新加载文章列表
   └─► renderFeed() 动态生成 HTML 卡片
   └─► 新文章出现在页面顶部

⑥ 其他访客打开页面
   └─► fetchPosts() 从 Supabase 拉取所有文章
   └─► renderFeed() 渲染文章列表
   └─► 看到刚刚发布的文章（但没有编辑按钮）
```

---

## 11. 技术选型 — 为什么这样设计？

### 11.1 为什么选 Serverless 而不是传统后端？

| 维度 | 传统后端（Flask/Express） | 本项目（Supabase + GitHub Pages） |
|:---|:---|:---|
| 💰 成本 | 服务器月租 ¥50-200 | 免费 |
| 🔧 运维 | 需要配置服务器、数据库、Nginx | 零运维 |
| 🚀 部署 | 每次改代码要重新部署后端 | 只需推一个 HTML 文件 |
| 📚 学习曲线 | 需要学 Python/Node + 数据库 + Linux | 只需 HTML/CSS/JS |
| 👤 适合谁 | 复杂业务、多用户系统 | 个人博客、小型项目 |

### 11.2 为什么用 Quill.js 而不是自己写编辑器？

自己写一个支持加粗、斜体、列表的编辑器，需要处理：
- `document.execCommand()` 或 `Selection API`
- 跨浏览器兼容
- 撤销/重做
- 粘贴过滤

Quill.js 已经解决了所有这些问题，而且**只有 43KB**（gzip 后）。

### 11.3 为什么用管理密码而不是真正的用户系统？

对于**个人博客**：
- 只有你一个人需要发文章
- 不需要用户注册、邮箱验证、密码重置
- 管理密码方案 = 10 行代码 vs 完整认证系统 = 100+ 行代码

但如果以后想做**多用户博客**，可以升级到 Supabase Auth。

---

## 📚 总结 — 你学到了什么

| 层次 | 知识点 | 在本项目中 |
|:---|:---|:---|
| HTML | 标签结构、`id`/`class`、`<header>`/`<aside>`/`<main>` | 页面骨架 |
| CSS | 变量、Grid 布局、响应式、动画、过渡 | 报纸风格、手机适配 |
| JavaScript | DOM 操作、`async/await`、模板字符串、事件处理 | 全部交互逻辑 |
| 数据库 | PostgreSQL、SQL 的 CRUD、Supabase JS SDK | 文章存储与查询 |
| API | REST 概念、HTTP 方法、JSON 数据格式 | 前后端通信 |
| 认证 | anon key、JWT 概念、localStorage | 管理密码方案 |
| 部署 | Git、GitHub Pages、CDN | 免费上线 |
| 架构 | Serverless、客户端渲染、Fluent Interface | 整体设计思路 |

> 🎯 **一个 HTML 文件，连接一个云端数据库，就构成了一个完整的博客系统。** 这就是现代 Web 开发的魅力。